In [0]:
spark.sql("SHOW VOLUMES IN workspace.ecommerce").show()

In [0]:
input_path = "/Volumes/workspace/ecommerce/silver_volume/stream_input"
output_path = "/Volumes/workspace/ecommerce/silver_volume/stream_output"
checkpoint_path = "/Volumes/workspace/ecommerce/silver_volume/stream_checkpoint"
output_table = "workspace.ecommerce.streaming_events_silver"

In [0]:
dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(output_path)
dbutils.fs.mkdirs(checkpoint_path)

print("Streaming paths ready")

In [0]:
source_data = "/Volumes/workspace/ecommerce/ecommerce_data"

dbutils.fs.cp(
    source_data + "/2019-Oct.csv",
    input_path + "/2019-Oct.csv"
)

print("File added to stream input")

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

In [0]:
stream_df = (
    spark.readStream
    .format("csv")
    .schema(schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", 1)   # Micro-batch simulation
    .load(input_path)
)

In [0]:
from pyspark.sql import functions as F

clean_stream_df = stream_df.filter(
    (F.col("user_id").isNotNull()) &
    (F.col("price") >= 0)
)

In [0]:
query = (
    clean_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)  
    .start(output_path)
)

query.awaitTermination()

In [0]:
spark.read.format("delta").load(output_path) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(output_table)

print("Streaming table created:", output_table)

In [0]:
spark.sql(f"""
SELECT event_type, COUNT(*) as total_events
FROM {output_table}
GROUP BY event_type
ORDER BY total_events DESC
""").show()

In [0]:
source_data = "/Volumes/workspace/ecommerce/ecommerce_data"

dbutils.fs.cp(
    source_data + "/2019-Oct.csv",
    input_path + "/2019-Oct.csv"
)

print("New file added to streaming folder")

In [0]:
query.status

In [0]:
spark.read.format("delta").load(output_path).display()